# 实验报告03：正则表达式与网络结构化数据

姓名：张梓望　　学号：20244081104　　班级：24 数信 01 班

## 一、实验环境与提交要求

### 1.1 第一个代码框：身份与环境记录

使用独立的 pytorch 虚拟环境，未使用 base 环境。

In [1]:
import hashlib
import os
import platform
import uuid
import sys
# 修改为自己的学号
STUDENT_ID = 20244081104
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境] pytorch
[学号]       20244081104
[机器指纹] 61a4c30073bfaca8
[RUN_ID]   20244081104-61a4c30073bfaca8
[Python]    3.11.13


## 二、实验内容与过程

### 实验1：正则表达式的基础匹配

`\d+` 匹配数字，`\s+` 匹配空白字符，`\bcat\b` 里的 `\b` 表示单词边界，所以 cats 中间的 cat 不会被算进去。

In [2]:
import re

text1 = "cat on mat; later: cats and dogs"
print("\\d+   ", re.findall(r"\d+", text1))
print("\\s+   ", re.findall(r"\s+", text1))
print("\\bcat\\b", re.findall(r"\bcat\b", text1))

\d+    []
\s+    [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b ['cat']


`|` 表示或者，正则引擎按分支的书写顺序逐个尝试。先写短分支，就会把本可以更长的匹配拆散。

In [3]:
text2 = "cat dog bird CAT"
print("cat|dog          ", re.findall(r"cat|dog", text2))
print("cat|dog 忽略大小写", re.findall(r"cat|dog", text2, flags=re.I))
print("a|aa|aaa         ", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a         ", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog           ['cat', 'dog']
cat|dog 忽略大小写 ['cat', 'dog', 'CAT']
a|aa|aaa          ['a', 'a', 'a', 'a']
aaa|aa|a          ['aaa', 'a']


五个量词分别控制 a 的重复次数：`*` 可以出现 0 次，`+` 至少 1 次，`?` 最多 1 次，`{2}` 恰好 2 次，`{1,2}` 在 1 到 2 次之间。

In [4]:
text3 = "ct cat caat caaat"
for p in (r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"):
    print(p, re.findall(p, text3))

ca*t ['ct', 'cat', 'caat', 'caaat']
ca+t ['cat', 'caat', 'caaat']
ca?t ['ct', 'cat']
ca{2}t ['caat']
ca{1,2}t ['cat', 'caat']


### 实验2：分组、回溯引用与替换

捕获组把邮箱拆成用户名和域名，`re.findall` 在有多个分组时返回元组列表。

In [5]:
text_email = "联系：alice@example.com；备选：bob@test.org"
result = re.findall(r"([A-Za-z0-9._%+-]+)@([A-Za-z0-9.-]+\.[A-Za-z]{2,})", text_email)
print(result)
for user, domain in result:
    print("用户名", user, "域名", domain)

[('alice', 'example.com'), ('bob', 'test.org')]
用户名 alice 域名 example.com
用户名 bob 域名 test.org


`\1` 引用第一个分组刚刚捕获的内容，所以能找出前后重复的单词。

In [6]:
text_dup = "cat cat, dog cat, bird bird"
print(re.findall(r"\b(\w+)\s+\1\b", text_dup))

['cat', 'bird']


三个捕获组依次是年、月、日，分隔符用字符类 `[-/.]` 覆盖三种写法。替换串写成 `\2-\3-\1`，把月提到最前、年放到最后。`re.sub` 只替换匹配到的部分，行数不会变。

In [7]:
text_date = "2024-05-01\n2024/05/02\n2024.05.03"
new_date = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", text_date)

print("转换前：")
print(text_date)
print()
print("转换后：")
print(new_date)
print()
print("行数", len(text_date.splitlines()), "->", len(new_date.splitlines()))

转换前：
2024-05-01
2024/05/02
2024.05.03

转换后：
05-01-2024
05-02-2024
05-03-2024

行数 3 -> 3


两种号码的分隔符不同，但位置一样，用同一个模式匹配之后重新拼成统一格式。

In [8]:
text_phone = "联系我：123-456-7890 或 987.654.3210"
print(re.sub(r"(\d{3})[-.](\d{3})[-.](\d{4})", r"(\1) \2-\3", text_phone))

联系我：(123) 456-7890 或 (987) 654-3210


### 实验3：使用 BeautifulSoup 解析 HTML

In [9]:
from bs4 import BeautifulSoup

html_doc = '''
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
'''

soup = BeautifulSoup(html_doc, "html.parser")

print("标题文本：", soup.title.text)
print("h1 标签名：", soup.h1.name)
print("h1 文本：", soup.h1.text)

标题文本： 实验课程
h1 标签名： h1
h1 文本： 数据分析


In [10]:
courses = soup.find_all("p", class_="course")
for p in courses:
    print(p.get_text())

links = soup.find_all("a")
for a in links:
    print(a.get_text(), a["href"])

第六讲：正则表达式
第七讲：结构化数据
讲义六 https://example.org/lecture06
讲义七 https://example.org/lecture07


In [11]:
print("整个文档：", repr(soup.get_text()))
print("单个段落：", repr(courses[0].get_text()))
print("单个链接：", repr(links[0].get_text()))

整个文档： '\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'
单个段落： '第六讲：正则表达式'
单个链接： '讲义六'


`get_text()` 只取标签内部的文字。整篇取时会带上标签之间的换行，单个标签取到的则是干净的字符串。